### TOPIC MODELING HOTELES EN ESPAÑOL

In [5]:
import pandas as pd
import os
from langdetect import detect, DetectorFactory

# Fijamos la semilla para que la detección sea siempre la misma
DetectorFactory.seed = 0

def es_espanol_o_vacio(texto):
    """
    Devuelve True si el texto está en español, si está vacío (NaN),
    o si no contiene texto detectable (ej. solo números o emojis).
    """
    if pd.isna(texto) or str(texto).strip() == "":
        return True
    
    try:
        idioma = detect(str(texto))
        return idioma == 'es'
    except:
        # Conservamos casos sin letras (ej. "10/10", "---")
        return True

# 1. Cargar el dataset limpio COMPLETO
print("Cargando el dataset completo...")
df_limpio = pd.read_csv('datos/procesados/df_comentarios_limpio.csv')
total_inicial = len(df_limpio)
print(f"Total de comentarios iniciales cargados: {total_inicial}")

# 2. Aplicar la detección de idioma a todo el dataset
# Nota: Si el dataset es muy grande, este paso puede tardar un par de minutos
print("\nAnalizando idiomas de los comentarios... (Esto puede tardar un poco)")
mask_pos = df_limpio['positivo'].apply(es_espanol_o_vacio)
mask_neg = df_limpio['negativo'].apply(es_espanol_o_vacio)

# 3. Filtrar el dataset (nos quedamos con los que sean español, nulo o indetectable en ambas columnas)
df_espanol = df_limpio[mask_pos & mask_neg]
total_espanol = len(df_espanol)
print(f"-> Comentarios retenidos en español: {total_espanol} (Se descartaron {total_inicial - total_espanol} en otros idiomas)")

# 4. Crear carpeta y guardar los mini datasets por hotel
carpeta_destino = 'mini_datasets_hoteles'
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)

columna_hotel = 'nombre_del_hotel'

print(f"\nAgrupando por hotel y guardando archivos en la carpeta '{carpeta_destino}'...")
for nombre_hotel, df_hotel in df_espanol.groupby(columna_hotel):
    
    nombre_archivo_seguro = str(nombre_hotel).replace(' ', '_').replace('/', '-')
    ruta_archivo = f"{carpeta_destino}/comentarios_{nombre_archivo_seguro}.csv"
    
    # Guardamos el mini dataset del hotel
    df_hotel.to_csv(ruta_archivo, index=False)
    
print(f"¡Listo! Se han guardado {df_espanol[columna_hotel].nunique()} mini datasets procesados y filtrados por idioma.")

Cargando el dataset completo...
Total de comentarios iniciales cargados: 58720

Analizando idiomas de los comentarios... (Esto puede tardar un poco)
-> Comentarios retenidos en español: 35844 (Se descartaron 22876 en otros idiomas)

Agrupando por hotel y guardando archivos en la carpeta 'mini_datasets_hoteles'...
¡Listo! Se han guardado 12 mini datasets procesados y filtrados por idioma.
